Ради интереса, можно посмотреть посредством перебора, какие таблицы с последовательными годами будут получаться у разных рынков при разном наборе параметров.

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from scipy import stats

In [2]:
markets = {
    'Germany': ['SIE.DE', 'VOW.DE', 'MBG.DE', 'BMW.DE', 'TKA.DE', 'DHL.DE',
              'LHA.DE', 'BAS.DE', 'BAYN.DE', 'MRK.DE', 'FRE.DE', 'FME.DE',
              'ALV.DE', 'MUV2.DE', 'DBK.DE', 'CBK.DE', 'ADS.DE', 'BEI.DE',
              'HEN3.DE', 'SAP.DE', 'DTE.DE', 'IFX.DE', 'EOAN.DE', 'RWE.DE', 'CON.DE'],

    'USA': ['GE', 'F', 'NUE', 'UPS', 'DD', 'PFE', 'JNJ', 'DVA',
              'BRK-B', 'AIG', 'C', 'BAC', 'NKE', 'PG', 'CL', 'MSFT',
              'T', 'INTC', 'XOM', 'CVX', 'BWA', 'WMT', 'KO', 'MCD'],

    'UK': ['BP', 'SHEL.L', 'HSBA.L', 'LLOY.L', 'BARC.L', 'ANTO.L', 'RIO.L',
              'AAL.L', 'ULVR.L', 'DGE.L', 'AZN.L', 'GSK.L', 'VOD.L', 'TSCO.L',
              'RR.L', 'PRU.L', 'AV.L', 'LSEG.L', 'REL.L', 'ABF.L',
              'SBRY.L', 'STAN.L', 'LGEN.L', 'III.L'],

    'Japan': ['7203.T', '7267.T', '7201.T', '6954.T', '6758.T', '6702.T',
              '6501.T', '6971.T', '8316.T', '8411.T', '7751.T',
              '6752.T', '4502.T', '4503.T', '8058.T', '8031.T', '9501.T',
              '9983.T', '8267.T', '9432.T', '7011.T', '7731.T', '7733.T','6504.T'],

    'China': ['000002.SZ', '000063.SZ', '000858.SZ', '600000.SS', '600011.SS',
              '600016.SS', '600019.SS', '600028.SS', '600029.SS', '600030.SS',
              '600036.SS', '600050.SS', '600104.SS', '600115.SS', '600887.SS',
              '600188.SS', '600256.SS', '600406.SS', '600519.SS', '600690.SS'
            ]
}

START_YEAR = 2006
END_YEAR = 2012
gamma_values = [0.2, 0.3, 0.4, 0.5, 0.6]
conf_levels = [0.9, 0.95, 0.99]

In [3]:
# Скачиваем данные и вычисляем log-доходности
def get_log_returns(tickers, start=f'{START_YEAR}-01-01', end=f'{END_YEAR+1}-01-01'):
    prices = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)['Close']
    returns = np.log(prices / prices.shift(1)).dropna(axis=0)
    return returns

In [4]:
# 10 наименее коррелированных акций за все года
def get_top_low_correlated(tickers, n_top=10):
    returns = get_log_returns(tickers)

    # средняя абсолютная корреляция для каждой акции
    all_abs_corrs = {asset: [] for asset in tickers}
    for year in returns.index.year.unique():
        df_year = returns[returns.index.year == year]
        corr_year = df_year.corr()

        for asset in tickers:
            corr_series = corr_year[asset].drop(asset, errors='ignore')
            all_abs_corrs[asset].extend(corr_series.abs().tolist())

    avg_corrs = {}
    for asset in tickers:
        avg_corrs[asset] = np.mean(all_abs_corrs[asset])

    # берём топ-10
    sorted_assets = sorted(avg_corrs.items(), key=lambda x: x[1])
    top_10 = [asset for asset, i in sorted_assets[:10]]

    return top_10

In [5]:
# Округление пика распределения корреляций в меньшую сторону
def get_peak_threshold(returns, bins=75, round_to=0.05):
    corr_list = []

    for year in range(START_YEAR, END_YEAR+1):#####
        year_data = returns[returns.index.year == year]
        if len(year_data) > 0:
            corr_mat = year_data.corr()
            upper_tri = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
            corr_list.append(upper_tri)

    all_corrs = np.array(corr_list).flatten()
    all_corrs_clean = all_corrs[~np.isnan(all_corrs)]

    hist, bin_edges = np.histogram(all_corrs_clean, bins=bins)
    peak_bin = np.argmax(hist)
    peak = bin_edges[peak_bin]

    threshold = np.floor(peak / round_to) * round_to

    print(f"Пик распределения: {peak:.4f}, округлённый порог: {threshold:.2f}")
    return threshold

In [6]:
# Статистические тесты
def run_two_sample_tests(returns, years, gamma_values, conf_levels, market_name):
    asset_names = returns.columns.tolist()
    n_assets = len(asset_names)
    M = n_assets * (n_assets - 1) // 2

    # Рассчет статистик
    def get_test_T_statistic(returns_subset, gamma0):
        n_obs = len(returns_subset)
        corr_mat = returns_subset.corr()
        T_mat = pd.DataFrame(np.nan, index=asset_names, columns=asset_names)

        for i in range(n_assets):
            for j in range(i + 1, n_assets):
                asset_i = asset_names[i]
                asset_j = asset_names[j]

                if asset_i in corr_mat.index and asset_j in corr_mat.index:
                    r_ij = corr_mat.loc[asset_i, asset_j]
                    if not np.isnan(r_ij):
                        T = np.sqrt(n_obs - 1) * (r_ij - gamma0) / np.sqrt(1 - r_ij**2)
                        T_mat.loc[asset_i, asset_j] = T
                        T_mat.loc[asset_j, asset_i] = T
        return T_mat

    # Построение множеств значимых ребер и значимых неребер
    def get_significant_sets(T_mat, c_quantile):
        Le = []
        Ln = []

        for i in range(T_mat.shape[0]):
            for j in range(i + 1, T_mat.shape[1]):
                T_val = T_mat.iloc[i, j]
                if not np.isnan(T_val):
                    if T_val > c_quantile:
                        Le.append((T_mat.index[i], T_mat.columns[j]))
                    elif T_val < -c_quantile:
                        Ln.append((T_mat.index[i], T_mat.columns[j]))

        return set(Le), set(Ln)

    # Тесты
    def two_sample_test(data1, data2, c_quantile, gamma0):
        T1 = get_test_T_statistic(data1, gamma0)
        T2 = get_test_T_statistic(data2, gamma0)
        Le1, Ln1 = get_significant_sets(T1, c_quantile)
        Le2, Ln2 = get_significant_sets(T2, c_quantile)

        if (Ln1 & Le2) or (Ln2 & Le1):
            return 1
        return 0

    for gamma0 in gamma_values:
      for conf_level in conf_levels:
        alpha = 1 - conf_level
        c_quantile = stats.norm.ppf(1 - alpha / M)
        print(f"gamma0 = {gamma0} and confidence level = {conf_level}")

        for i in range(len(years) - 1):
          year1 = years[i]
          year2 = years[i + 1]

          data1 = returns[returns.index.year == year1]
          data2 = returns[returns.index.year == year2]

          result = two_sample_test(data1, data2, c_quantile, gamma0)
          period = f"{year1}-{year2}"
          print(f"{period}: {result}")

In [7]:
for market_name, tickers in markets.items():
    print(f"\n{'#'*60}")
    print(f"# Рынок: {market_name}")
    print(f"{'#'*60}")

    returns_full = get_log_returns(tickers)
    top_tickers = get_top_low_correlated(tickers, n_top=10)
    returns = returns_full[top_tickers]
    years = sorted(returns.index.year.unique())
    print(f"Годы: {years}")

    peak = get_peak_threshold(returns)
    print(f"peak is {peak}")
    gamma_values.append(peak)

    run_two_sample_tests(returns, years, gamma_values, conf_levels, market_name)
    gamma_values.remove(peak)



############################################################
# Рынок: Germany
############################################################
Годы: [2006, 2007, 2008, 2009, 2010, 2011, 2012]
Пик распределения: 0.2468, округлённый порог: 0.20
peak is 0.2
gamma0 = 0.2 and confidence level = 0.9
2006-2007: 0
2007-2008: 0
2008-2009: 0
2009-2010: 0
2010-2011: 0
2011-2012: 0
gamma0 = 0.2 and confidence level = 0.95
2006-2007: 0
2007-2008: 0
2008-2009: 0
2009-2010: 0
2010-2011: 0
2011-2012: 0
gamma0 = 0.2 and confidence level = 0.99
2006-2007: 0
2007-2008: 0
2008-2009: 0
2009-2010: 0
2010-2011: 0
2011-2012: 0
gamma0 = 0.3 and confidence level = 0.9
2006-2007: 0
2007-2008: 1
2008-2009: 0
2009-2010: 0
2010-2011: 1
2011-2012: 0
gamma0 = 0.3 and confidence level = 0.95
2006-2007: 0
2007-2008: 1
2008-2009: 0
2009-2010: 0
2010-2011: 0
2011-2012: 0
gamma0 = 0.3 and confidence level = 0.99
2006-2007: 0
2007-2008: 0
2008-2009: 0
2009-2010: 0
2010-2011: 0
2011-2012: 0
gamma0 = 0.4 and confidence level = 